# Ticket Category Router — Companion Notebook

**What this notebook does:** mirrors `customer-support-assistant/ticket_category.py` cell by cell — the assistant's decision-tree classifier that routes a support ticket to the right team (**shipping**, **billing**, or **account**). This is the ticket-category capability first promised in Lesson 11, built in Lesson 16.

The tree is grown from 12 labelled tickets: at every step it greedily picks the yes/no word question that best un-mixes the categories (lowest Gini impurity). The project keeps a single, readable tree; the Lesson 16 notebook additionally shows the random-forest upgrade (many trees voting) that a later lesson can bring into the project.

In [ ]:
# past tickets a human already labelled with the team that handled them
TRAINING_TICKETS = [
    ("Where is my package it has not arrived", "shipping"),
    ("My order says delivered but no package came", "shipping"),
    ("The delivery is late and tracking has not updated", "shipping"),
    ("My package arrived damaged and crushed", "shipping"),
    ("I was charged twice for my order please refund me", "billing"),
    ("I want a refund for this purchase", "billing"),
    ("Why was I charged extra fees I want a refund", "billing"),
    ("Requesting a refund because the discount was not applied", "billing"),
    ("My login is not working after the update", "account"),
    ("I forgot my password and the reset email never comes", "account"),
    ("Please help me change the email on my account", "account"),
    ("My account is locked and I need to reset my password", "account"),
]

print("Training tickets:", len(TRAINING_TICKETS))

## The questions the tree may ask

The tree can only ask yes/no questions of the form *"does the ticket contain the word W?"*. `CANDIDATE_WORDS` is the menu; the tree decides on its own which questions matter and in what order. Questions are about **exact** words after simple lowercasing and punctuation-stripping — "delivery" and "delivered" are different words to this model.

In [ ]:
# the menu of yes/no questions: "does the ticket contain this word?"
CANDIDATE_WORDS = [
    "refund", "charged", "package", "delivery", "order",
    "password", "account", "login", "help",
]


def tokenize(text):
    # lowercase, strip basic punctuation, split on spaces
    lowered = text.lower().replace(",", "").replace(".", "").replace("!", "").replace("?", "")
    return lowered.split()


def has_word(text, word):
    # True if the ticket contains this exact word
    return word in tokenize(text)


# demo on one ticket
print(tokenize("I was charged twice for my order please refund me"))
print(has_word("I was charged twice for my order please refund me", "refund"))

## Measuring messiness and choosing the best question

Three small functions do all the learning:

- **`gini`** — how mixed a pile of labels is: the chance that two tickets drawn at random from the pile disagree. 0 means pure, 0.667 is our maximally mixed three-way pile.
- **`split_quality`** — a question's score: split the pile into yes/no, then take the weighted average of the two sides' Gini values. Lower is better. Questions that put every ticket on one side return `None` and are skipped.
- **`best_word`** — try every question on the menu, keep the lowest score.

The full walkthrough of the arithmetic is in the Lesson 16 notebook; here we just confirm the winner at the root is "refund" (score 0.333 — one cut isolates all four billing tickets).

In [ ]:
def gini(labels):
    total = len(labels)
    # count how many times each label appears
    counts = {}
    for label in labels:
        if label not in counts:
            counts[label] = 0
        counts[label] += 1
    # start from 1 and subtract each label's squared share
    impurity = 1.0
    for label in counts:
        share = counts[label] / total
        impurity -= share * share
    return impurity


def split_quality(tickets, word):
    # sort every ticket's label onto the yes side or the no side
    yes_labels = []
    no_labels = []
    for text, label in tickets:
        if has_word(text, word):
            yes_labels.append(label)
        else:
            no_labels.append(label)
    # a question that puts everything on one side tells us nothing
    if len(yes_labels) == 0 or len(no_labels) == 0:
        return None
    # weighted average of the two sides' impurities (lower = better)
    total = len(tickets)
    yes_weight = len(yes_labels) / total
    no_weight = len(no_labels) / total
    return yes_weight * gini(yes_labels) + no_weight * gini(no_labels)


def best_word(tickets):
    best = None
    best_score = None
    for word in CANDIDATE_WORDS:
        score = split_quality(tickets, word)
        if score is None:
            continue                      # useless question here, skip it
        if best_score is None or score < best_score:
            best = word                   # new best question so far
            best_score = score
    return best


# score every question on the full 12 tickets and show the winner
print("Scores at the root (lower = better):")
for word in CANDIDATE_WORDS:
    score = split_quality(TRAINING_TICKETS, word)
    print("  " + word + ":", round(score, 3))

print()
print("Best first question:", best_word(TRAINING_TICKETS))

## Growing the tree

`build_tree` asks the best question, splits the tickets into a yes pile and a no pile, then calls itself on each pile — recursion. It stops and places a leaf (predicting the pile's majority label) when the pile is pure, the depth limit `MAX_DEPTH` is reached, or no question splits the pile.

The depth limit is the overfitting control: without it, the tree would keep splitting until every leaf was pure, memorizing individual tickets instead of learning patterns (Lesson 13's warning, in tree form).

In [ ]:
# depth limit: how many questions deep the tree may grow
MAX_DEPTH = 3


def majority_label(tickets):
    # count how many tickets carry each label
    counts = {}
    for text, label in tickets:
        if label not in counts:
            counts[label] = 0
        counts[label] += 1
    # find the label with the highest count
    winner = None
    winner_count = 0
    for label in counts:
        if counts[label] > winner_count:
            winner = label
            winner_count = counts[label]
    return winner


def build_tree(tickets, depth=0):
    # collect the labels in this pile
    labels = []
    for text, label in tickets:
        labels.append(label)
    # stop if the pile is pure or the depth limit is reached
    if gini(labels) == 0.0 or depth == MAX_DEPTH:
        return {"label": majority_label(tickets)}
    # find the best question for this pile
    word = best_word(tickets)
    # stop if no question splits this pile at all
    if word is None:
        return {"label": majority_label(tickets)}
    # split the pile into the yes side and the no side
    yes_tickets = []
    no_tickets = []
    for text, label in tickets:
        if has_word(text, word):
            yes_tickets.append((text, label))
        else:
            no_tickets.append((text, label))
    # recurse: grow a subtree on each side, one level deeper
    return {
        "word": word,
        "yes": build_tree(yes_tickets, depth + 1),
        "no": build_tree(no_tickets, depth + 1),
    }


def format_tree(node, indent=""):
    # a leaf just states its prediction
    if "label" in node:
        return "predict " + node["label"]
    # a question node shows its question, then both branches indented below
    yes_part = format_tree(node["yes"], indent + "  ")
    no_part = format_tree(node["no"], indent + "  ")
    return ('contains "' + node["word"] + '"?'
            + "\n" + indent + "  yes: " + yes_part
            + "\n" + indent + "  no:  " + no_part)


# grow the tree from the training tickets and print it
TREE = build_tree(TRAINING_TICKETS)
print("Learned decision tree:")
print(format_tree(TREE))

## Predicting: walk a ticket down the tree

To classify a new ticket, start at the root question and answer yes or no; each answer chooses a branch, and the walk ends at a leaf, whose label is the prediction. In the module this is `predict_category`, the function `main.py` imports.

Note the built-in default: a ticket that mentions **none** of the tree's chosen words falls all the way down the no-branches into the `account` leaf. The tree does not understand "account-ness" — that leaf is simply where the leftovers of the training data landed.

In [ ]:
def predict_category(ticket_text):
    node = TREE
    # keep answering questions until we land on a leaf
    while "label" not in node:
        if has_word(ticket_text, node["word"]):
            node = node["yes"]    # answer was yes: follow the yes branch
        else:
            node = node["no"]     # answer was no: follow the no branch
    return node["label"]          # the leaf's label is the prediction


# route three new tickets, one per category
demo_tickets = [
    "Where is my package it has been two weeks",
    "I was charged twice and want a refund",
    "I cannot log in please reset my password",
]
for ticket in demo_tickets:
    print("  [" + predict_category(ticket) + "]", ticket)

## Self-check

The same checks as the module's `if __name__ == "__main__":` block: one clear ticket per category must route correctly, and the tree must classify all 12 training tickets correctly. If a future change to the training data or the tree code breaks any of these, the `assert` lines will raise an error instead of failing silently.

In [ ]:
# one clear ticket from each category must be routed to the right team
assert predict_category("Where is my package it has been two weeks") == "shipping"
assert predict_category("I was charged twice and want a refund") == "billing"
assert predict_category("I cannot log in please reset my password") == "account"

# every training ticket must be classified correctly
correct = 0
for text, label in TRAINING_TICKETS:
    if predict_category(text) == label:
        correct += 1
assert correct == len(TRAINING_TICKETS)

print("Self-check passed.")